# Phase 131 — CryptoBench: Weighted Cross-Scale Ensemble

Phase130 equal-weighted all 3 scales (1/3 each) despite 3B-transfer being
clearly the strongest (val PR/MCC) and 650M-scratch the weakest. Grid-
searches ensemble weights on VAL ONLY, applies the winning weights to
TEST once. Zero new training -- pure recombination of Phase130's
predictions.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q torch-geometric==2.6.1

## Cell 2 — Model (identical to Phase130)

In [ ]:
import json, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import (average_precision_score, roc_auc_score,
                              matthews_corrcoef, f1_score)

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")
WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)

class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)

class LocalizingEGNN(nn.Module):
    def __init__(s, n_pre, td, h=128, L=4, dr=0.1, use_loc_head=True):
        super().__init__(); s.n_pre=n_pre; s.use_loc_head=use_loc_head
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        if use_loc_head:
            s.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))
    def forward(s, d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits = s.head(h).squeeze(-1)
        pred_center = None
        if s.use_loc_head:
            batch = d.batch if hasattr(d, "batch") and d.batch is not None \
                    else torch.zeros(h.size(0), dtype=torch.long, device=h.device)
            loc_logit = s.loc_head(h).squeeze(-1)
            attn = pyg_softmax(loc_logit, batch)
            weighted = attn.unsqueeze(-1) * pos
            n_graphs = int(batch.max().item()) + 1
            pred_center = torch.zeros(n_graphs, 3, device=h.device).index_add_(0, batch, weighted)
        return logits, pred_center

@torch.no_grad()
def predict_seed(model, loader):
    """Returns pooled (probs, y) in loader order. Alignment across scales
    is verified separately, per-CHAIN (not per-residue) directly from the
    unbatched dataset list -- PyG's DataLoader collates non-tensor
    attributes like pdb_id/chain_id per-GRAPH, not per-NODE, so zipping
    them against per-residue probs inside the batched loop would silently
    misalign (length mismatch: one entry per chain vs one per residue)."""
    all_probs, all_y = [], []
    for b in loader:
        b = b.to(DEVICE)
        lg, pc = model(b)
        all_probs.append(torch.sigmoid(lg).cpu().numpy())
        all_y.append(b.y.cpu().numpy())
    return np.concatenate(all_probs), np.concatenate(all_y)

## Cell 3 — Load each scale's blob + 8 checkpoints, predict on VAL and TEST (identical to Phase130)

In [ ]:
SCALES = {
    "35M":  dict(blob="phase42_cryptobench_graphs.pt",       ckpt_prefix="cryptobench_transfer_seed"),
    "650M": dict(blob="phase45_cryptobench_graphs_esm650m.pt", ckpt_prefix="cryptobench_esm650m_deploy_seed"),
    "3B":   dict(blob="phase48_cryptobench_graphs_esm3b.pt",   ckpt_prefix="cryptobench_esm3b_transfer_seed"),
}

scale_val_probs, scale_val_y, scale_val_keys = {}, {}, {}
scale_test_probs, scale_test_y, scale_test_keys = {}, {}, {}
t0 = time.time()
for scale, cfg in SCALES.items():
    blob = torch.load(locate(cfg["blob"]), map_location="cpu", weights_only=False)
    dataset = blob["dataset"]
    N_PRE = int(blob["n_pre_topo"]); TOPO_DIM = int(blob["topo_dim"])
    val_set = [d for d in dataset if d.split == "val"]
    test_set = [d for d in dataset if d.split == "test"]
    val_loader = DataLoader(val_set, batch_size=16)
    test_loader = DataLoader(test_set, batch_size=16)
    scale_val_keys[scale] = [(d.pdb_id, d.chain_id, int(d.num_nodes)) for d in val_set]
    scale_test_keys[scale] = [(d.pdb_id, d.chain_id, int(d.num_nodes)) for d in test_set]

    val_seed_probs, test_seed_probs = [], []
    for seed in range(8):
        ckpt = locate(f"{cfg['ckpt_prefix']}{seed}.pt")
        m = LocalizingEGNN(N_PRE, TOPO_DIM).to(DEVICE)
        m.load_state_dict(torch.load(ckpt, map_location=DEVICE)); m.eval()
        vp, vy = predict_seed(m, val_loader)
        tp, ty = predict_seed(m, test_loader)
        val_seed_probs.append(vp); test_seed_probs.append(tp)
        if seed == 0:
            scale_val_y[scale] = vy; scale_test_y[scale] = ty
    scale_val_probs[scale] = np.mean(val_seed_probs, axis=0)
    scale_test_probs[scale] = np.mean(test_seed_probs, axis=0)
    print(f"  {scale:<6} 8-seed val+test predictions ready ({cfg['ckpt_prefix']}*)  [{(time.time()-t0)/60:.1f}m]")

for split_name, keys_dict in [("val", scale_val_keys), ("test", scale_test_keys)]:
    ref = keys_dict["35M"]
    for scale in ("650M", "3B"):
        assert keys_dict[scale] == ref, f"{split_name}: {scale} key order mismatch vs 35M"
print(f"\nalignment verified on both val and test across all 3 scales")

## Cell 4 — Grid-search ensemble weights on VAL ONLY, apply winning weights + val-tuned threshold to TEST

In [ ]:
def metrics_at(p, y, thresh):
    pred = (p >= thresh).astype(int)
    return {"auc": roc_auc_score(y, p), "auprc": average_precision_score(y, p),
            "mcc": matthews_corrcoef(y, pred), "f1": f1_score(y, pred)}

def best_thresh(p, y):
    best_t, best_mcc = 0.5, -1.0
    for t in np.arange(0.05, 0.96, 0.01):
        mcc = matthews_corrcoef(y, (p >= t).astype(int))
        if mcc > best_mcc: best_mcc, best_t = mcc, t
    return best_t, best_mcc

y_val_ref = scale_val_y["35M"]
y_test_ref = scale_test_y["35M"]
SCALE_NAMES = ["35M", "650M", "3B"]

# grid search: weights on a 0.1 simplex (w1+w2+w3=1, each a multiple of 0.1)
best_w, best_val_mcc, best_val_auc_at_w = None, -1.0, None
grid_results = []
for w1 in np.arange(0.0, 1.01, 0.1):
    for w2 in np.arange(0.0, 1.01 - w1, 0.1):
        w3 = 1.0 - w1 - w2
        if w3 < -1e-9: continue
        w3 = max(w3, 0.0)
        weights = {"35M": w1, "650M": w2, "3B": w3}
        val_p = sum(weights[s] * scale_val_probs[s] for s in SCALE_NAMES)
        t, val_mcc = best_thresh(val_p, y_val_ref)
        val_auc = roc_auc_score(y_val_ref, val_p)
        grid_results.append(dict(w35m=w1, w650m=w2, w3b=w3, val_mcc=val_mcc, val_auc=val_auc, thresh=t))
        if val_mcc > best_val_mcc:
            best_val_mcc, best_w, best_val_auc_at_w = val_mcc, weights, val_auc

print(f"best weights by VAL MCC: 35M={best_w['35M']:.1f}  650M={best_w['650M']:.1f}  3B={best_w['3B']:.1f}")
print(f"  (val MCC {best_val_mcc:.4f}, val AUC {best_val_auc_at_w:.4f})")

# apply winning weights + a threshold re-tuned on val with those weights, to TEST
test_p_weighted = sum(best_w[s] * scale_test_probs[s] for s in SCALE_NAMES)
val_p_weighted = sum(best_w[s] * scale_val_probs[s] for s in SCALE_NAMES)
final_thresh, _ = best_thresh(val_p_weighted, y_val_ref)
weighted_result = metrics_at(test_p_weighted, y_test_ref, final_thresh)

# also grid-search separately by VAL AUC (threshold-independent), in case it picks different weights
best_w_auc, best_val_auc = None, -1.0
for r in grid_results:
    weights = {"35M": r["w35m"], "650M": r["w650m"], "3B": r["w3b"]}
    if r["val_auc"] > best_val_auc:
        best_val_auc, best_w_auc = r["val_auc"], weights
test_p_auc = sum(best_w_auc[s] * scale_test_probs[s] for s in SCALE_NAMES)
t_auc, _ = best_thresh(sum(best_w_auc[s] * scale_val_probs[s] for s in SCALE_NAMES), y_val_ref)
weighted_result_auc = metrics_at(test_p_auc, y_test_ref, t_auc)

print(f"\n{'variant':<28}{'AUC':>8}{'AUPRC':>8}{'MCC':>8}{'F1':>8}")
print("-"*60)
print(f"{'equal-weight (Phase130)':<28}{0.9140:>8.4f}{0.4311:>8.4f}{0.4530:>8.4f}{0.4776:>8.4f}")
print(f"{'weighted (val-MCC-optimal)':<28}{weighted_result['auc']:>8.4f}{weighted_result['auprc']:>8.4f}{weighted_result['mcc']:>8.4f}{weighted_result['f1']:>8.4f}")
print(f"{'weighted (val-AUC-optimal)':<28}{weighted_result_auc['auc']:>8.4f}{weighted_result_auc['auprc']:>8.4f}{weighted_result_auc['mcc']:>8.4f}{weighted_result_auc['f1']:>8.4f}")
print(f"{'3B-transfer alone (best single)':<28}{0.9111:>8.4f}{0.4307:>8.4f}{0.4664:>8.4f}{0.4896:>8.4f}")

json.dump({
    "best_weights_by_val_mcc": best_w, "val_mcc": best_val_mcc,
    "best_weights_by_val_auc": best_w_auc, "val_auc": best_val_auc,
    "test_result_val_mcc_weights": weighted_result,
    "test_result_val_auc_weights": weighted_result_auc,
    "grid": grid_results,
}, open(WORK/"phase131_weighted_ensemble_results.json", "w"), indent=2)
print("\nsaved -> phase131_weighted_ensemble_results.json")

---

## Reading this

- **Weighted beats both equal-weight ensemble AND 3B-transfer alone**
  -- the best possible outcome, a genuinely optimized combination worth
  adopting as the final CryptoBench number.
- **Weighted beats equal-weight but not 3B-transfer alone** -- the
  ensemble adds a little value but 650M is still net-diluting; consider
  dropping it and using only 35M+3B (or 3B alone).
- **Weighted ties equal-weight** -- the equal-weight combination was
  already close to optimal, nothing left on the table here.